# 실습 2. 로지스틱 회귀로 피싱사이트 분류

**과목:** 인공지능과 정보보호 | **데이터:** Web page phishing detection dataset  
**목표:** 피싱 URL의 숫자형 특성을 이용해 정상(0)/피싱(1)을 예측한다. 중요 계수와 예측 확률을 분석하고, **임계값 0.3·0.5·0.8**의 오탐(FP)·미탐(FN)을 비교한다.

- 강의자료: `7week_02___URL...` 로지스틱 회귀와 확률·임계값 실습
- 데이터 출처: https://www.kaggle.com/datasets/shashwatwork/web-page-phishing-detection-dataset
- Google Colab에서 위에서부터 실행한다. 실제 정확도와 오탐/미탐 수는 데이터 실행 결과에서 확인한다.

> 실행 환경: Google Colab (Python 3). 외부 CSV를 원본 데이터셋에서 다운로드하고 학습·평가를 실행한다.

In [ ]:
# 공개 Kaggle 데이터셋 다운로드를 위한 도구를 코랩에 설치한다.
%pip -q install kagglehub

In [ ]:
# 파일 경로 관리와 표 데이터를 다루기 위한 라이브러리를 불러온다.
from pathlib import Path
import pandas as pd
import numpy as np
import kagglehub

# 강의자료가 지정한 피싱 URL 데이터셋과 CSV 파일명을 사용한다.
DATASET_ID = 'shashwatwork/web-page-phishing-detection-dataset'
EXPECTED_CSV = 'dataset_phishing.csv'
try:
    data_folder = Path(kagglehub.dataset_download(DATASET_ID))
    csv_files = list(data_folder.rglob('*.csv'))
    matched = [p for p in csv_files if p.name.lower() == EXPECTED_CSV.lower()]
    csv_path = matched[0] if matched else (csv_files[0] if len(csv_files)==1 else None)
    if csv_path is None:
        raise FileNotFoundError(f'원하는 CSV 파일을 찾을 수 없습니다: {csv_files}')
except Exception as err:
    # Kaggle 자동 다운로드가 실패하면 원본 CSV 업로드로 이어갈 수 있게 한다.
    print('자동 다운로드 실패:', type(err).__name__, str(err)[:220])
    print('Kaggle에서 dataset_phishing.csv를 받아 업로드하세요.')
    from google.colab import files
    uploaded = files.upload()
    csv_path = Path(next((n for n in uploaded if n.lower().endswith('.csv')), ''))
    if not csv_path.is_file():
        raise FileNotFoundError('dataset_phishing.csv 업로드가 필요합니다.')

# 원본 URL 데이터셋과 정답 분포를 확인한다.
url = pd.read_csv(csv_path)
print('데이터 파일:', csv_path.name)
print('원본 크기(행, 열):', url.shape)
print('원본 정답 수:', url['status'].value_counts().to_dict())
print('열 이름 일부:', list(url.columns[:12]))

## 1. 데이터 전처리와 80:20 분할

피싱사이트 원본 문자열 `url`은 수업에서 다루는 숫자형 피처 모델의 입력에서 제외한다. `status`는 legitimate=0, phishing=1로 바꾼다.

In [ ]:
# legitimate=0(정상), phishing=1(피싱)으로 레이블을 통일한다.
labels = url['status'].astype(str).str.strip().str.lower()
target = labels.map({'legitimate': 0, 'phishing': 1})
if target.isna().any():
    raise ValueError(f'알 수 없는 status 값: {sorted(labels[target.isna()].unique())}')
target = target.astype(int)

# 원본 URL 문자 열과 정답 열을 학습 입력에서 제외한다.
# 수업에서는 이미 추출된 숫자형 URL 피처를 학습에 사용한다.
data = url.drop(columns=['url', 'status'], errors='ignore')
# 숫자처럼 저장된 문자열도 숫자형으로 바꾸고, 변환 불가능한 값은 NaN으로 표시한다.
data = data.apply(pd.to_numeric, errors='coerce')
# 숫자형 값이 하나도 없는 열은 학습할 정보가 없어 제외한다.
empty_cols = data.columns[data.isna().all()].tolist()
if empty_cols:
    print('전체 결측 열 제외:', empty_cols)
    data = data.drop(columns=empty_cols)

print('입력 피처 크기:', data.shape, '/ 레이블:', target.value_counts().to_dict())
print('결측값 총 개수:', int(data.isna().sum().sum()))
assert data.shape[1] > 0, '학습에 사용할 숫자 피처가 없습니다.'

# 수업과 같이 80%는 훈련, 20%는 테스트에 사용한다.
# 층화 분할(stratify)로 각 클래스의 비율을 비슷하게 유지한다.
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42, stratify=target
)
print('훈련:', train_input.shape, '테스트:', test_input.shape)

## 2. 로지스틱 회귀 학습·예측·정확도

숫자 피처의 단위가 다를 수 있으므로 결측 보정과 표준화 후 로지스틱 회귀를 학습한다. 전처리는 훈련 세트에 대해서만 `fit`되도록 Pipeline에 넣었다.

In [ ]:
# 필요한 분류 모델과 전처리·평가 함수를 불러온다.
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# 훈련 중앙값 대체 → 표준화 → 로지스틱 회귀로 하나의 모델을 만든다.
lr = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)
# 훈련 데이터로만 모델의 가중치와 전처리 통계를 학습한다.
lr.fit(train_input, train_target)
# 테스트 사이트를 정상(0) 또는 피싱(1)으로 예측한다.
predict = lr.predict(test_input)
# 강의의 accuracy_score는 예측값과 정답값을 비교해야 한다.
print('훈련 정확도:', f'{lr.score(train_input,train_target):.4f}')
print('테스트 정확도:', f'{accuracy_score(test_target,predict):.4f}')
print('정밀도(피싱):', f'{precision_score(test_target,predict):.4f}')
print('재현율(피싱):', f'{recall_score(test_target,predict):.4f}')
print('F1(피싱):', f'{f1_score(test_target,predict):.4f}')
print('혼동행렬 [실제 0/1 × 예측 0/1]:\n', confusion_matrix(test_target,predict,labels=[0,1]))

## 3. 피싱 판단에 기여한 계수 살펴보기

계수가 양수이면 **다른 특성이 같은 조건에서** 피싱(1) 방향으로, 음수이면 정상(0) 방향으로 영향을 준다. 단일 피처의 계수만으로 실제 공격 원인을 단정하지 않는다.

In [ ]:
# 표준화된 피처에 대한 로지스틱 회귀 계수를 열 이름과 연결한다.
coef = pd.Series(lr.named_steps['logisticregression'].coef_[0], index=data.columns)
# 양의 계수 상위 10개와 음의 계수 하위 10개를 나누어 확인한다.
print('[피싱(1) 방향 계수 상위 10개]')
display(coef.sort_values(ascending=False).head(10).to_frame('coefficient'))
print('[정상(0) 방향 계수 상위 10개]')
display(coef.sort_values(ascending=True).head(10).to_frame('coefficient'))
# 전체 피처 중 절댓값이 가장 큰 계수도 확인한다.
print('절댓값이 큰 특성 10개:', coef.abs().sort_values(ascending=False).head(10).index.tolist())

# 강의자료에 나온 주요 URL 피처가 실제 데이터에 있으면 각 계수를 점검한다.
for f in ['length_url','nb_dots','nb_hyphens','nb_at','https_token','ip']:
    if f in coef.index:
        direction = '피싱 쪽' if coef[f]>0 else '정상 쪽'
        print(f'{f}: {coef[f]:+.4f} → {direction} (다른 변수 동일 조건)')

## 4. 피싱 확률과 임계값: 0.3·0.5·0.8

- **오탐(FP):** 정상사이트(0)를 피싱(1)으로 잘못 차단
- **미탐(FN):** 피싱사이트(1)를 정상(0)으로 놓침
- 낮은 임계값은 보통 피싱을 더 많이 잡지만 정상사이트도 더 많이 차단할 수 있다.

In [ ]:
# predict_proba의 1번 열은 status=phishing(1)의 예측 확률이다.
phishing_prob = lr.predict_proba(test_input)[:, 1]
print('처음 5개 URL의 피싱 확률:', np.round(phishing_prob[:5],4))

# 임계값별 오탐과 미탐 개수를 계산한다.
threshold_results = []
for threshold in [0.3, 0.5, 0.8]:
    pred_at_threshold = (phishing_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_target,pred_at_threshold,labels=[0,1]).ravel()
    threshold_results.append({
        '임계값': threshold,
        '정확도': accuracy_score(test_target,pred_at_threshold),
        '정밀도': precision_score(test_target,pred_at_threshold,zero_division=0),
        '재현율': recall_score(test_target,pred_at_threshold,zero_division=0),
        '오탐(FP)': int(fp),
        '미탐(FN)': int(fn),
        'TP': int(tp), 'TN': int(tn)
    })
threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))

# 시험에서 읽기 쉽게 임계값별 오탐·미탐을 막대그래프로 표시한다.
threshold_df.plot(x='임계값',y=['오탐(FP)','미탐(FN)'],kind='bar',rot=0,figsize=(7,4))
plt.ylabel('Number of websites')
plt.title('Threshold trade-off: FP vs FN')
plt.tight_layout()
plt.show()

## 5. 어떤 임계값을 선택할까? (3줄 해석)

금융서비스처럼 피싱 놓침(미탐)을 더 위험하게 보는 정책이면 낮은 임계값을 우선 고려할 수 있다. 반대로 정상 접속 차단(오탐)이 큰 피해를 초래하면 임계값을 높일 수 있다. 아래 셀은 실제 실행된 숫자로 세 줄 설명을 만든다.

In [ ]:
# 정책 예시: 미탐 최소화 → 오탐 최소화 순서로 세 후보 중 임계값을 선택한다.
choice = threshold_df.sort_values(['미탐(FN)','오탐(FP)','임계값']).iloc[0]
low = threshold_df.loc[threshold_df['임계값']==0.3].iloc[0]
high = threshold_df.loc[threshold_df['임계값']==0.8].iloc[0]
print(f"1. 0.3일 때 오탐 {int(low['오탐(FP)'])}건·미탐 {int(low['미탐(FN)'])}건, "
      f"0.8일 때 오탐 {int(high['오탐(FP)'])}건·미탐 {int(high['미탐(FN)'])}건이다.")
print(f"2. 이번 실험에서 미탐을 최소화하는 정책의 후보로 임계값 {choice['임계값']:.1f}을 선택한다.")
print('3. 다만 오탐으로 정상사이트가 차단될 수 있으므로 실제 배포 시 피해 비용과 서비스 요구에 맞춰 재검증한다.')

## 6. 직접 점검

1. `accuracy_score(test_target, predict)`처럼 **정답과 예측값**을 비교했는지 확인한다. 강의자료의 일부 예시 코드에는 변수 오타가 있으므로 수정하였다.
2. 피처 계수는 **표준화한 피처** 기준이므로 크기를 비교할 수 있지만 인과관계를 의미하지 않는다.
3. `status`: 0=정상, 1=피싱을 모든 혼동행렬에서 동일하게 유지한다.
4. 코랩 실행 결과를 확인한 뒤 자신의 표현으로 해석을 점검한다.